# Duplicidade de Leads e Otimização de Agendamentos
## CRM Hospital São Rafael — Análise Independente

Pedro Henrique Batista Bergamin, 559443
José Cláudio da Silva Júnior , 559680
Evandro Yuji Kaibara de Oliveira, 559274
Caio Martinez Saes, 560753
Estefany Caetano de Jesus, 560181

---

### Contexto do Sistema

O módulo de **Agendamento** (`AgendamentoEntity`) do CRM São Rafael conecta:

- `PacienteEntity` → campos identificadores: **nome, cpf, email, telefone**
- `MedicoEntity`   → disponibilidade de horários no dia
- `EtapaEntity` / `StatusEntity` → estado do agendamento

Quando novos leads chegam (potenciais pacientes), dois problemas surgem:

1. **Duplicidade de cadastro** — o lead pode já existir no banco com dados parcialmente diferentes.
2. **Encaixe de agenda** — os slots livres do médico precisam ser aproveitados ao máximo.

---

### Objetivos

| # | Problema | Técnica |
|---|----------|---------|
| 1 | Verificação recursiva de duplicidade | Recursão pura |
| 2 | Comparação por combinações de campos (cpf, email, telefone, nome) | `itertools.combinations` |
| 3 | Evitar recomputar comparações entre pares já analisados | Memoização (`lru_cache`) |
| 4 | Melhor encaixe de consultas nos slots do médico | Recursão + DP com dicionário de memo |

In [1]:
from __future__ import annotations

import functools
import time as time_module
from dataclasses import dataclass
from datetime import datetime, date
from itertools import combinations
from typing import Optional

print("Imports OK")

Imports OK


---
## Parte 1 — Deduplicação de Leads

### Estrutura de Dados

Baseada nos campos de `PacienteEntity` que identificam unicamente um paciente:

| Campo | Tipo | Unicidade no banco |
|-------|------|-------------------|
| `nome` | `String(100)` | — |
| `cpf`  | `String(14)` | `unique=true` |
| `email`| `String(150)` | `unique=true` |
| `telefone` | `String(20)` | — |

Leads chegam com dados **parcialmente preenchidos**. Cadastros existentes têm dados completos.

> `frozen=True` torna as dataclasses **hashable**, necessário para uso com `lru_cache`.

In [2]:
@dataclass(frozen=True)
class Lead:
    """Representa um novo lead recebido (dados possivelmente incompletos)."""
    nome: str
    cpf: Optional[str] = None
    email: Optional[str] = None
    telefone: Optional[str] = None


@dataclass(frozen=True)
class Cadastro:
    """Representa um paciente já registrado no banco de dados."""
    id: int
    nome: str
    cpf: Optional[str] = None
    email: Optional[str] = None
    telefone: Optional[str] = None


# ---------- Dados de exemplo ----------
cadastros_existentes: list[Cadastro] = [
    Cadastro(id=1, nome="Ana Paula Silva",   cpf="123.456.789-00", email="ana@email.com",     telefone="(11) 91234-5678"),
    Cadastro(id=2, nome="Carlos Mendes",     cpf="234.567.890-11", email="carlos@email.com",  telefone="(21) 98765-4321"),
    Cadastro(id=3, nome="Fernanda Oliveira", cpf="345.678.901-22", email="fer@email.com",     telefone="(31) 97654-3210"),
    Cadastro(id=4, nome="Roberto Lima",      cpf="456.789.012-33", email="roberto@email.com", telefone="(41) 96543-2109"),
    Cadastro(id=5, nome="Mariana Costa",     cpf="567.890.123-44", email="mari@email.com",    telefone="(51) 95432-1098"),
    Cadastro(id=6, nome="Joao Pedro Santos", cpf="678.901.234-55", email="joao@email.com",    telefone="(61) 94321-0987"),
    Cadastro(id=7, nome="Patricia Rocha",    cpf="789.012.345-66", email="pati@email.com",    telefone="(71) 93210-9876"),
    Cadastro(id=8, nome="Lucas Barbosa",     cpf="890.123.456-77", email="lucas@email.com",   telefone="(81) 92109-8765"),
]

novos_leads: list[Lead] = [
    Lead(nome="Ana Paula Silva",   cpf="123.456.789-00"),                           # duplicata exata   → id=1
    Lead(nome="Carlos M.",         email="carlos@email.com"),                        # duplicata e-mail  → id=2
    Lead(nome="Fernanda Oliveira", telefone="(99) 00000-0000"),                     # duplicata nome    → id=3
    Lead(nome="Novo Paciente X",   cpf="999.999.999-99", email="novo@email.com"),   # novo cadastro
    Lead(nome="Roberto Lima",      cpf="456.789.012-33", email="roberto@email.com"),# duplicata forte   → id=4
]

print(f"Cadastros no banco: {len(cadastros_existentes)}")
print(f"Novos leads para verificar: {len(novos_leads)}")

Cadastros no banco: 8
Novos leads para verificar: 5


### 1.1 — Verificação Recursiva de Duplicidade

A função percorre a lista de cadastros **recursivamente** (sem loop `for` explícito), verificando se algum campo identificador bate com o lead.

- **Caso base**: índice ultrapassou o final da lista → nenhuma duplicata encontrada.
- **Passo recursivo**: compara o cadastro atual; se coincide, retorna `(True, id)`; senão, avança para `index + 1`.

In [3]:
def verificar_duplicidade_recursiva(
    lead: Lead,
    cadastros: list[Cadastro],
    index: int = 0,
) -> tuple[bool, Optional[int]]:
    """
    Verifica recursivamente se um lead já existe na lista de cadastros.

    Compara CPF, e-mail, telefone e nome (nesta ordem de confiabilidade).
    Para no primeiro cadastro que coincida em qualquer campo.

    Args:
        lead:      lead a ser verificado
        cadastros: lista de cadastros existentes (banco de dados simulado)
        index:     posição atual na recursão — não passar externamente

    Returns:
        (encontrado, id_cadastro)  quando encontrou duplicata
        (False, None)              quando não encontrou
    """
    # Caso base: percorreu todos os cadastros sem encontrar
    if index >= len(cadastros):
        return False, None

    cad = cadastros[index]

    # Qualquer campo identificador coincidente → duplicata
    if (
        (lead.cpf      and lead.cpf      == cad.cpf)                        or
        (lead.email    and lead.email    == cad.email)                       or
        (lead.telefone and lead.telefone == cad.telefone)                    or
        (lead.nome.strip().lower()       == cad.nome.strip().lower())
    ):
        return True, cad.id

    # Passo recursivo: avança para o próximo cadastro
    return verificar_duplicidade_recursiva(lead, cadastros, index + 1)


# --- Demonstração ---
print("=" * 60)
print("Verificacao recursiva de duplicidade")
print("=" * 60)

for lead in novos_leads:
    encontrado, cad_id = verificar_duplicidade_recursiva(lead, cadastros_existentes)
    if encontrado:
        print(f"  '{lead.nome}': DUPLICATA  → cadastro id={cad_id}")
    else:
        print(f"  '{lead.nome}': NOVO CADASTRO")

Verificacao recursiva de duplicidade
  'Ana Paula Silva': DUPLICATA  → cadastro id=1
  'Carlos M.': DUPLICATA  → cadastro id=2
  'Fernanda Oliveira': DUPLICATA  → cadastro id=3
  'Novo Paciente X': NOVO CADASTRO
  'Roberto Lima': DUPLICATA  → cadastro id=4


### 1.2 — Análise por Combinações de Campos

A verificação simples (seção anterior) para no **primeiro** campo que corresponde.
Aqui exploramos **todas as combinações possíveis** de campos e atribuímos um **score de confiança**:

- Score 1 → correspondência fraca (ex.: apenas nome)
- Score 2+ → correspondência forte (ex.: CPF + e-mail)

Usa `itertools.combinations` para gerar os subconjuntos `{cpf}`, `{email}`, `{cpf, email}`, `{nome, telefone, email}`, etc.

In [ ]:
def _campos_coincidentes(lead: Lead, cad: Cadastro) -> list[str]:
    """Retorna quais campos identificadores coincidem entre lead e cadastro."""
    matches: list[str] = []
    if lead.cpf      and lead.cpf      == cad.cpf:      matches.append("cpf")
    if lead.email    and lead.email    == cad.email:    matches.append("email")
    if lead.telefone and lead.telefone == cad.telefone: matches.append("telefone")
    if lead.nome.strip().lower()       == cad.nome.strip().lower(): matches.append("nome")
    return matches


def buscar_por_combinacoes(
    lead: Lead,
    cadastros: list[Cadastro],
    min_campos: int = 1,
) -> list[dict]:
    """
    Busca cadastros que correspondam ao lead em pelo menos `min_campos` campos.

    Gera todas as combinações de tamanho 1..4 dos campos identificadores e,
    para cada combinação, verifica quais cadastros satisfazem todos os campos
    daquela combinação. Retorna resultados únicos ordenados por score decrescente.

    Args:
        lead:       lead a verificar
        cadastros:  lista de cadastros existentes
        min_campos: mínimo de campos coincidentes para considerar duplicata (padrão=1)

    Returns:
        Lista de dicts: {cadastro_id, nome_cadastro, campos_coincidentes, score, nivel}
    """
    CAMPOS = ("cpf", "email", "telefone", "nome")
    vistos: set[int] = set()
    resultados: list[dict] = []

    for r in range(min_campos, len(CAMPOS) + 1):
        for combo in combinations(CAMPOS, r):
            for cad in cadastros:
                if cad.id in vistos:
                    continue
                matches = _campos_coincidentes(lead, cad)
                if all(c in matches for c in combo):
                    vistos.add(cad.id)
                    resultados.append({
                        "cadastro_id":         cad.id,
                        "nome_cadastro":       cad.nome,
                        "campos_coincidentes": matches,
                        "score":               len(matches),
                        "nivel":               "FORTE" if len(matches) >= 2 else "FRACO",
                    })

    return sorted(resultados, key=lambda x: x["score"], reverse=True)


# --- Teste ---
print("=" * 60)
print("Busca por combinacoes de campos")
print("=" * 60)

for lead in novos_leads:
    resultados = buscar_por_combinacoes(lead, cadastros_existentes, min_campos=1)
    if resultados:
        print(f"\n  Lead '{lead.nome}':")
        for r in resultados:
            print(f"    id={r['cadastro_id']} | {r['nome_cadastro']!r} | "
                  f"campos={r['campos_coincidentes']} | score={r['score']} [{r['nivel']}]")
    else:
        print(f"\n  Lead '{lead.nome}': nenhuma correspondencia")

Busca por combinacoes de campos

  Lead 'Ana Paula Silva':
    id=1 | 'Ana Paula Silva' | campos=['cpf', 'nome'] | score=2 [FORTE]

  Lead 'Carlos M.':
    id=2 | 'Carlos Mendes' | campos=['email'] | score=1 [FRACO]

  Lead 'Fernanda Oliveira':
    id=3 | 'Fernanda Oliveira' | campos=['nome'] | score=1 [FRACO]

  Lead 'Novo Paciente X': nenhuma correspondencia → NOVO CADASTRO

  Lead 'Roberto Lima':
    id=4 | 'Roberto Lima' | campos=['cpf', 'email', 'nome'] | score=3 [FORTE]


### 1.3 — Memoização para Evitar Comparações Repetidas

Quando múltiplos leads chegam em lote (ex.: importação de planilha), a mesma comparação `(lead, cadastro)` pode ser solicitada mais de uma vez.  
`@functools.lru_cache` armazena o resultado de cada par calculado em memória.

- **1ª rodada**: todas as comparações são calculadas e armazenadas no cache (*cache miss*).
- **Rodadas seguintes**: o mesmo par retorna instantaneamente do cache (*cache hit*).

Como `Lead` e `Cadastro` são `@dataclass(frozen=True)`, são **hashable** e podem ser chave de cache.

In [ ]:
@functools.lru_cache(maxsize=None)
def _score_par(lead: Lead, cadastro: Cadastro) -> int:
    """
    Calcula o score de correspondência entre um lead e um cadastro.

    Memoizado: se consultado novamente com o mesmo par, retorna do cache.

    Pontuação:
        +1  CPF coincidente
        +1  e-mail coincidente
        +1  telefone coincidente
        +1  nome coincidente (case-insensitive, sem espaços extras)
    """
    score = 0
    if lead.cpf      and lead.cpf      == cadastro.cpf:               score += 1
    if lead.email    and lead.email    == cadastro.email:             score += 1
    if lead.telefone and lead.telefone == cadastro.telefone:          score += 1
    if lead.nome.strip().lower()       == cadastro.nome.strip().lower(): score += 1
    return score


def verificar_duplicidade_memoizada(
    lead: Lead,
    cadastros: list[Cadastro],
    limiar: int = 1,
) -> list[dict]:
    """
    Detecta duplicatas usando comparação memoizada.

    Benefício em lote: o custo de _score_par(lead, cad) é pago apenas
    na primeira vez; chamadas repetidas com o mesmo par são O(1).

    Args:
        lead:      lead a verificar
        cadastros: lista de cadastros existentes
        limiar:    score mínimo para considerar duplicata (padrão=1)

    Returns:
        Lista ordenada por score de cadastros que superam o limiar
    """
    resultados: list[dict] = []
    for cad in cadastros:
        score = _score_par(lead, cad)
        if score >= limiar:
            resultados.append({
                "cadastro_id":   cad.id,
                "nome_cadastro": cad.nome,
                "score":         score,
                "nivel":         "FORTE" if score >= 2 else "FRACO",
            })
    return sorted(resultados, key=lambda x: x["score"], reverse=True)


# --- Teste: 3 rodadas sobre os mesmos leads ---
print("=" * 60)
print("Verificacao memoizada — processamento em lote (3 rodadas)")
print("=" * 60)

_score_par.cache_clear() 

for rodada in range(1, 4):
    inicio = time_module.perf_counter()
    for lead in novos_leads:
        verificar_duplicidade_memoizada(lead, cadastros_existentes)
    elapsed_ms = (time_module.perf_counter() - inicio) * 1_000
    info = _score_par.cache_info()
    print(f"  Rodada {rodada}: {elapsed_ms:.4f} ms | "
          f"hits={info.hits:>3}, misses={info.misses:>3}, cache_size={info.currsize}")

print()
print("Resultados finais (última rodada):")
for lead in novos_leads:
    resultados = verificar_duplicidade_memoizada(lead, cadastros_existentes)
    if resultados:
        melhor = resultados[0]
        print(f"  '{lead.nome}'  →  DUPLICATA id={melhor['cadastro_id']} "
              f"[{melhor['nivel']}] score={melhor['score']}")
    else:
        print(f"  '{lead.nome}'  →  NOVO CADASTRO")

Verificacao memoizada — processamento em lote (3 rodadas)
  Rodada 1: 0.1073 ms | hits=  0, misses= 40, cache_size=40
  Rodada 2: 0.0384 ms | hits= 40, misses= 40, cache_size=40
  Rodada 3: 0.0336 ms | hits= 80, misses= 40, cache_size=40

Resultados finais (última rodada):
  'Ana Paula Silva'  →  DUPLICATA id=1 [FORTE] score=2
  'Carlos M.'  →  DUPLICATA id=2 [FRACO] score=1
  'Fernanda Oliveira'  →  DUPLICATA id=3 [FRACO] score=1
  'Novo Paciente X'  →  NOVO CADASTRO
  'Roberto Lima'  →  DUPLICATA id=4 [FORTE] score=3


---
## Parte 2 — Otimização de Agenda com Subproblemas

### Contexto

No sistema real, `AgendamentoEntity` associa um `PacienteEntity` a um `MedicoEntity` em um `data_hora` específico.  
Um médico possui **slots de tempo livre** ao longo do dia (de durações variadas). Novas consultas chegam com uma **duração necessária**.

### Problema

> **Dado** um conjunto de slots disponíveis e uma lista de solicitações de consulta,  
> **encontrar** a atribuição que **maximiza o total de minutos agendados** no dia.

Este é um problema análogo ao **Knapsack 0-1**:  
- Um slot só pode receber **uma** consulta.  
- Uma consulta só é alocada em um slot com duração **suficiente**.

Sem memoização, a recursão revisitaria o mesmo estado `(índice_consulta, slots_já_usados)` múltiplas vezes.  
Com memoização, cada estado é calculado **uma única vez**.

In [ ]:
@dataclass(frozen=True)
class Slot:
    """Intervalo de tempo livre na agenda do médico."""
    inicio: datetime
    fim: datetime

    @property
    def duracao(self) -> int:
        """Duração do slot em minutos."""
        return int((self.fim - self.inicio).total_seconds() // 60)

    def __str__(self) -> str:
        return f"{self.inicio.strftime('%H:%M')}–{self.fim.strftime('%H:%M')} ({self.duracao}min)"


@dataclass(frozen=True)
class SolicitacaoConsulta:
    """Solicitação de agendamento de um paciente."""
    paciente_id: int
    paciente_nome: str
    duracao_min: int   # duração necessária em minutos


# ---------- Dados mocados de teste ----------
def _dt(h: int, m: int) -> datetime:
    return datetime(2026, 3, 30, h, m)

slots_do_dia: list[Slot] = [
    Slot(_dt( 8,  0), _dt( 8, 30)),   # Slot 0 — 30 min
    Slot(_dt( 8, 40), _dt( 9, 40)),   # Slot 1 — 60 min
    Slot(_dt(10,  0), _dt(10, 20)),   # Slot 2 — 20 min
    Slot(_dt(10, 30), _dt(11, 30)),   # Slot 3 — 60 min
    Slot(_dt(13,  0), _dt(13, 45)),   # Slot 4 — 45 min
    Slot(_dt(14,  0), _dt(14, 30)),   # Slot 5 — 30 min
    Slot(_dt(15,  0), _dt(16,  0)),   # Slot 6 — 60 min
]

solicitacoes: list[SolicitacaoConsulta] = [
    SolicitacaoConsulta(101, "Ana Paula Silva",    30),
    SolicitacaoConsulta(102, "Carlos Mendes",      60),
    SolicitacaoConsulta(103, "Fernanda Oliveira",  45),
    SolicitacaoConsulta(104, "Roberto Lima",       20),
    SolicitacaoConsulta(105, "Mariana Costa",      60),
    SolicitacaoConsulta(106, "Joao Pedro Santos",  30),
    SolicitacaoConsulta(107, "Patricia Rocha",     90),  # nao caberá em nenhum slot
]

print("Slots disponíveis do médico em 30/03/2026:")
total_disp = 0
for i, s in enumerate(slots_do_dia):
    print(f"  Slot {i}: {s}")
    total_disp += s.duracao

print(f"\nTotal disponível: {total_disp} min\n")

print("Solicitacoes de consulta:")
for sol in solicitacoes:
    print(f"  Paciente {sol.paciente_id} ({sol.paciente_nome}): {sol.duracao_min} min")

Slots disponíveis do médico em 30/03/2026:
  Slot 0: 08:00–08:30 (30min)
  Slot 1: 08:40–09:40 (60min)
  Slot 2: 10:00–10:20 (20min)
  Slot 3: 10:30–11:30 (60min)
  Slot 4: 13:00–13:45 (45min)
  Slot 5: 14:00–14:30 (30min)
  Slot 6: 15:00–16:00 (60min)

Total disponível: 305 min

Solicitacoes de consulta:
  Paciente 101 (Ana Paula Silva): 30 min
  Paciente 102 (Carlos Mendes): 60 min
  Paciente 103 (Fernanda Oliveira): 45 min
  Paciente 104 (Roberto Lima): 20 min
  Paciente 105 (Mariana Costa): 60 min
  Paciente 106 (Joao Pedro Santos): 30 min
  Paciente 107 (Patricia Rocha): 90 min


### 2.1 — Algoritmo Recursivo com Memoização

**Estado** memoizado: `(índice_da_consulta_atual, frozenset_de_slots_já_usados)`

Para cada consulta, o algoritmo explora duas opções:
1. **Pular** (não agendar a consulta).
2. **Encaixar** em cada slot disponível que tenha duração suficiente.

A opção que gerar o maior total de minutos agendados é escolhida.  
`frozenset` é usado para o conjunto de slots usados por ser **imutável e hashable** (chave de dicionário válida).

In [ ]:
def _otimizar_recursivo(
    slots_dur: tuple[int, ...],
    consultas_dur: tuple[int, ...],
    idx: int,
    usados: frozenset[int],
    memo: dict,
) -> tuple[int, list[tuple[int, int]]]:
    """
    Recursão com memoização para maximizar minutos agendados no dia.

    Args:
        slots_dur:     durações de cada slot disponível (tuple imutável → hashable)
        consultas_dur: durações necessárias de cada consulta (tuple imutável → hashable)
        idx:           índice da consulta que está sendo processada agora
        usados:        frozenset dos índices de slots já atribuídos
        memo:          dicionário de memoização  {(idx, usados) → resultado}

    Returns:
        (total_min_agendados, [(idx_consulta, idx_slot), ...])
    """
    chave = (idx, usados)
    if chave in memo:
        return memo[chave]       


    if idx >= len(consultas_dur):
        memo[chave] = (0, [])
        return 0, []

    duracao_necessaria = consultas_dur[idx]


    melhor_min, melhor_atrib = _otimizar_recursivo(
        slots_dur, consultas_dur, idx + 1, usados, memo
    )


    for i, dur_slot in enumerate(slots_dur):
        if i in usados:
            continue                  
        if dur_slot < duracao_necessaria:
            continue            

        novos_usados = usados | frozenset([i])
        resto_min, resto_atrib = _otimizar_recursivo(
            slots_dur, consultas_dur, idx + 1, novos_usados, memo
        )
        total = duracao_necessaria + resto_min
        if total > melhor_min:
            melhor_min  = total
            melhor_atrib = [(idx, i)] + resto_atrib

    memo[chave] = (melhor_min, melhor_atrib)
    return melhor_min, melhor_atrib


def otimizar_agenda(
    slots: list[Slot],
    consultas: list[SolicitacaoConsulta],
) -> dict:
    """
    Interface pública: encontra a melhor atribuição de consultas a slots.

    Converte listas em tuples (hashable) antes de chamar a recursão.

    Returns:
        dict com estatísticas e lista detalhada de atribuições
    """
    memo: dict = {}
    slots_dur     = tuple(s.duracao      for s in slots)
    consultas_dur = tuple(c.duracao_min  for c in consultas)

    total_min, atribuicoes = _otimizar_recursivo(
        slots_dur, consultas_dur, 0, frozenset(), memo
    )

    total_disp = sum(slots_dur)
    resultado = {
        "total_minutos_agendados": total_min,
        "total_slots_disponiveis": total_disp,
        "aproveitamento_pct":      round(total_min / total_disp * 100, 1) if total_disp else 0,
        "estados_memoizados":      len(memo),
        "atribuicoes":             [],
        "nao_agendados":           [],
    }

    consultados_idx: set[int] = set()
    for consulta_idx, slot_idx in atribuicoes:
        sol = consultas[consulta_idx]
        slt = slots[slot_idx]
        consultados_idx.add(consulta_idx)
        resultado["atribuicoes"].append({
            "paciente_id":        sol.paciente_id,
            "paciente":           sol.paciente_nome,
            "duracao_solicitada": sol.duracao_min,
            "slot":               str(slt),
            "folga_min":          slt.duracao - sol.duracao_min,
        })

    for i, sol in enumerate(consultas):
        if i not in consultados_idx:
            resultado["nao_agendados"].append(sol.paciente_nome)

    return resultado


# --- Execução ---
resultado = otimizar_agenda(slots_do_dia, solicitacoes)

print("=" * 60)
print("Resultado da otimizacao de agenda")
print("=" * 60)
print(f"  Total agendado:      {resultado['total_minutos_agendados']} min")
print(f"  Total disponivel:    {resultado['total_slots_disponiveis']} min")
print(f"  Aproveitamento:      {resultado['aproveitamento_pct']}%")
print(f"  Estados memoizados:  {resultado['estados_memoizados']}")
print()
print("Atribuicoes:")
for a in resultado["atribuicoes"]:
    print(f"  {a['paciente']:<22} -> {a['slot']:<28} "
          f"solicitado={a['duracao_solicitada']}min  folga={a['folga_min']}min")

if resultado["nao_agendados"]:
    print()
    print("Nao agendados (sem slot compativel):")
    for nome in resultado["nao_agendados"]:
        print(f"  [sem slot] {nome}")

Resultado da otimizacao de agenda
  Total agendado:      245 min
  Total disponivel:    305 min
  Aproveitamento:      80.3%
  Estados memoizados:  521

Atribuicoes:
  Ana Paula Silva        -> 08:00–08:30 (30min)          solicitado=30min  folga=0min
  Carlos Mendes          -> 08:40–09:40 (60min)          solicitado=60min  folga=0min
  Fernanda Oliveira      -> 10:30–11:30 (60min)          solicitado=45min  folga=15min
  Roberto Lima           -> 10:00–10:20 (20min)          solicitado=20min  folga=0min
  Mariana Costa          -> 15:00–16:00 (60min)          solicitado=60min  folga=0min
  Joao Pedro Santos      -> 13:00–13:45 (45min)          solicitado=30min  folga=15min

Nao agendados (sem slot compativel):
  [sem slot] Patricia Rocha


### 2.2 — Comparação: Com Memoização × Sem Memoização

A versão sem memoização reexplora o mesmo estado `(idx, usados)` múltiplas vezes.  
O número de chamadas cresce exponencialmente com a quantidade de slots e consultas.

In [ ]:
def _otimizar_sem_memo(
    slots_dur: tuple[int, ...],
    consultas_dur: tuple[int, ...],
    idx: int,
    usados: frozenset[int],
    contador: list[int],
) -> tuple[int, list]:
    """
    Versão SEM memoização — usada apenas para comparação de chamadas recursivas.
    Identica à versão com memo, mas sem armazenar resultados intermediários.
    """
    contador[0] += 1 

    if idx >= len(consultas_dur):
        return 0, []

    duracao_necessaria = consultas_dur[idx]
    melhor_min, melhor_atrib = _otimizar_sem_memo(
        slots_dur, consultas_dur, idx + 1, usados, contador
    )

    for i, dur_slot in enumerate(slots_dur):
        if i in usados or dur_slot < duracao_necessaria:
            continue
        novos_usados = usados | frozenset([i])
        resto_min, resto_atrib = _otimizar_sem_memo(
            slots_dur, consultas_dur, idx + 1, novos_usados, contador
        )
        total = duracao_necessaria + resto_min
        if total > melhor_min:
            melhor_min  = total
            melhor_atrib = [(idx, i)] + resto_atrib

    return melhor_min, melhor_atrib


# --- Comparação ---
slots_dur     = tuple(s.duracao     for s in slots_do_dia)
consultas_dur = tuple(c.duracao_min for c in solicitacoes)

# Com memoização
memo = {}
t0 = time_module.perf_counter()
_otimizar_recursivo(slots_dur, consultas_dur, 0, frozenset(), memo)
t_memo = (time_module.perf_counter() - t0) * 1_000

# Sem memoização
contador = [0]
t0 = time_module.perf_counter()
_otimizar_sem_memo(slots_dur, consultas_dur, 0, frozenset(), contador)
t_sem = (time_module.perf_counter() - t0) * 1_000

print("=" * 60)
print("Comparacao: com memoizacao x sem memoizacao")
print("=" * 60)
print(f"  Com memoizacao  — estados unicos calculados : {len(memo):>6}  | tempo: {t_memo:.3f} ms")
print(f"  Sem memoizacao  — total de chamadas recursivas: {contador[0]:>6}  | tempo: {t_sem:.3f} ms")

reducao = (1 - len(memo) / max(contador[0], 1)) * 100
print(f"\n  Reducao de chamadas: {reducao:.1f}% gracas a memoizacao")

---
## Resumo

| Problema | Abordagem | Benefício |
|----------|-----------|-----------|
| Duplicidade de leads | Recursão por índice na lista | Sem loop explícito; fácil de estender |
| Combinações de campos | `itertools.combinations` | Avalia todos os subconjuntos de campos (cpf+email, email+telefone, etc.) |
| Memoização de pares | `@lru_cache` em `(Lead, Cadastro)` | Evita recomputar o mesmo par em lotes; custo amortizado O(1) |
| Encaixe de agenda | Recursão + DP — estado `(idx, frozenset_usados)` | Solução ótima garantida; memoização elimina reexploração exponencial |

---

### Limitações / Extensoes para producao

- **Similaridade de nomes**: substituir comparacao exata por distancia de Levenshtein ou fonetica (`rapidfuzz`, `jellyfish`)
- **CPF normalizado**: remover formatacao antes de comparar — `re.sub(r'\D', '', cpf)`
- **Escala**: para bases grandes, indexar por CPF/e-mail no banco elimina O(N) por lead
- **Recursao profunda**: para N de slots grande, substituir por DP iterativa (bottom-up) para evitar `RecursionError`
- **Integracao**: as funcoes de deduplicacao podem servir de base para um servico de pre-validacao no `PacienteService`

### parte deste notebook irá para o projeto final com serviços auxiliares como duckling, isso servirá para o chatbot, atuando como um metodos auxiliar para nosso agente de cadastro e agendamento

## Sprint 4 - Grafos e Dijkstra no CRM

**Objetivo:** modelar o fluxo do CRM como um grafo e encontrar o melhor caminho entre etapas do processo usando Dijkstra.

### Tarefa 1 - Representar o fluxo como grafo
Vamos transformar o fluxo do CRM em um **grafo direcionado e ponderado**, onde:
- cada etapa do funil e um no
- cada transicao e uma aresta direcionada
- cada aresta possui um custo (tempo/esforco operacional)

### Tarefa 2 - Implementar Dijkstra
Criar uma funcao para encontrar o menor caminho entre **Lead -> Confirmacao**.

### Tarefa 3 - Interpretar o resultado
Explicar:
- qual foi o menor caminho encontrado
- qual foi o custo total
- por que esse fluxo e mais eficiente

In [1]:
import heapq


def dijkstra(grafo, inicio, fim):
    """Retorna o menor custo e o caminho entre dois nos em um grafo direcionado."""
    fila = [(0, inicio)]
    distancias = {no: float("inf") for no in grafo}
    anteriores = {no: None for no in grafo}
    distancias[inicio] = 0

    while fila:
        custo_atual, no_atual = heapq.heappop(fila)

        if custo_atual > distancias[no_atual]:
            continue

        if no_atual == fim:
            break

        for vizinho, peso in grafo[no_atual].items():
            novo_custo = custo_atual + peso
            if novo_custo < distancias[vizinho]:
                distancias[vizinho] = novo_custo
                anteriores[vizinho] = no_atual
                heapq.heappush(fila, (novo_custo, vizinho))

    if distancias[fim] == float("inf"):
        return float("inf"), []

    caminho = []
    atual = fim
    while atual is not None:
        caminho.append(atual)
        atual = anteriores[atual]
    caminho.reverse()

    return distancias[fim], caminho


# Grafo direcionado do fluxo CRM (custos hipoteticos de tempo/esforco)
grafo_crm = {
    "Lead": {"Contato Inicial": 2, "Lead Perdido": 9},
    "Contato Inicial": {"Qualificacao": 1, "Sem Resposta": 4},
    "Sem Resposta": {"Contato Inicial": 2, "Lead Perdido": 3},
    "Qualificacao": {"Proposta": 2, "Lead Desqualificado": 5},
    "Proposta": {"Negociacao": 1, "Lead Perdido": 4},
    "Negociacao": {"Confirmacao": 1, "Lead Perdido": 3},
    "Confirmacao": {},
    "Lead Perdido": {},
    "Lead Desqualificado": {}
}

In [2]:
origem = "Lead"
destino = "Confirmacao"

custo_total, melhor_caminho = dijkstra(grafo_crm, origem, destino)

if melhor_caminho:
    print("Melhor caminho encontrado:")
    print(" -> ".join(melhor_caminho))
    print(f"Custo total: {custo_total}")
else:
    print(f"Nao existe caminho de {origem} para {destino}.")

Melhor caminho encontrado:
Lead -> Contato Inicial -> Qualificacao -> Proposta -> Negociacao -> Confirmacao
Custo total: 7


### Interpretacao do resultado (Tarefa 3)

Com os pesos definidos, o Dijkstra tende a retornar o fluxo:

**Lead -> Contato Inicial -> Qualificacao -> Proposta -> Negociacao -> Confirmacao**

- **Custo total esperado:** 7
- Esse caminho e mais eficiente porque evita transicoes com maior custo operacional (como retrabalho por "Sem Resposta" ou saidas para "Lead Perdido").
- Na pratica, ele representa um funil com menos desvios e menor tempo medio ate a confirmacao.

> Observacao: se os custos mudarem (por exemplo, por mudanca de SLA ou produtividade da equipe), o menor caminho pode ser diferente. Isso permite usar o modelo para simulacoes de melhoria de processo.

---
## Outros cenários de Grafo Direcionado e Ponderado + Dijkstra

O algoritmo de Dijkstra resolve o problema de **menor caminho** em qualquer grafo com pesos não-negativos.
Os três exemplos abaixo mostram aplicações fora do CRM, reutilizando a mesma função `dijkstra` já implementada.

| Cenário | Nós | Arestas (peso) |
|---------|-----|----------------|
| Transferência intra-hospitalar | Setores do hospital | Tempo de deslocamento (min) |
| Logística de entrega | Cidades / centros de distribuição | Distância (km) |
| Grade curricular | Disciplinas | Carga horária acumulada de pré-requisitos (h) |

### Cenário 1 — Transferência Intra-Hospitalar

Um paciente precisa ser transferido da **Triagem** até a **UTI**.
O grafo modela os setores do hospital e o **tempo médio de deslocamento em minutos** entre eles,
considerando corredores, elevadores e disponibilidade de maca.

Objetivo: encontrar a rota mais rápida para minimizar o tempo de transferência.

In [3]:
# Grafo dos setores do Hospital São Rafael
# Pesos = tempo médio de deslocamento em minutos
grafo_hospital = {
    "Triagem":          {"Pronto Socorro": 3, "Recepcao": 2},
    "Recepcao":         {"Triagem": 2, "Radiologia": 5, "Ambulatorio": 4},
    "Pronto Socorro":   {"Sala Cirurgia": 6, "UTI": 10, "Observacao": 3},
    "Observacao":       {"Pronto Socorro": 3, "UTI": 7, "Enfermaria": 4},
    "Radiologia":       {"Pronto Socorro": 4, "Ambulatorio": 3},
    "Ambulatorio":      {"Recepcao": 4, "Enfermaria": 5},
    "Sala Cirurgia":    {"UTI": 4, "Recuperacao": 3},
    "Recuperacao":      {"Enfermaria": 5, "UTI": 6},
    "Enfermaria":       {"Alta": 2},
    "UTI":              {"Recuperacao": 8, "Alta": 15},
    "Alta":             {},
}

origem_h = "Triagem"
destino_h = "UTI"
custo_h, caminho_h = dijkstra(grafo_hospital, origem_h, destino_h)

print("=" * 60)
print("Cenario 1 — Transferencia Intra-Hospitalar")
print("=" * 60)
print(f"  Origem:  {origem_h}")
print(f"  Destino: {destino_h}")
print()
print("  Rota mais rapida:")
print("  " + " -> ".join(caminho_h))
print(f"  Tempo total: {custo_h} minutos")
print()

# Rota alternativa: Triagem -> Alta (fluxo completo)
custo_alta, caminho_alta = dijkstra(grafo_hospital, "Triagem", "Alta")
print(f"  (Para comparacao) Triagem -> Alta: {custo_alta} min | {' -> '.join(caminho_alta)}")

Cenario 1 — Transferencia Intra-Hospitalar
  Origem:  Triagem
  Destino: UTI

  Rota mais rapida:
  Triagem -> Pronto Socorro -> UTI
  Tempo total: 13 minutos

  (Para comparacao) Triagem -> Alta: 12 min | Triagem -> Pronto Socorro -> Observacao -> Enfermaria -> Alta


### Cenário 2 — Logística de Entregas entre Cidades

Uma transportadora precisa levar um pacote de **São Paulo** até **Brasília**.
O grafo representa cidades e rodovias com **distância em km**.
Dijkstra encontra a rota com menor quilometragem total.

In [4]:
# Grafo rodoviário simplificado — pesos em km
grafo_cidades = {
    "Sao Paulo":        {"Campinas": 100, "Sorocaba": 90,  "Ribeirao Preto": 315},
    "Campinas":         {"Sao Paulo": 100, "Ribeirao Preto": 225, "Uberlandia": 460},
    "Sorocaba":         {"Sao Paulo": 90,  "Curitiba": 370},
    "Ribeirao Preto":   {"Campinas": 225,  "Uberlandia": 220, "Goiania": 570},
    "Uberlandia":       {"Campinas": 460,  "Ribeirao Preto": 220, "Brasilia": 390, "Goiania": 170},
    "Goiania":          {"Ribeirao Preto": 570, "Uberlandia": 170, "Brasilia": 210},
    "Curitiba":         {"Sorocaba": 370,  "Florianopolis": 300},
    "Florianopolis":    {"Curitiba": 300},
    "Brasilia":         {},
}

origem_c = "Sao Paulo"
destino_c = "Brasilia"
custo_c, caminho_c = dijkstra(grafo_cidades, origem_c, destino_c)

print("=" * 60)
print("Cenario 2 — Logistica de Entregas")
print("=" * 60)
print(f"  Origem:  {origem_c}")
print(f"  Destino: {destino_c}")
print()
print("  Rota com menor distancia:")
print("  " + " -> ".join(caminho_c))
print(f"  Distancia total: {custo_c} km")
print()

# Comparar partindo de Campinas
custo_c2, caminho_c2 = dijkstra(grafo_cidades, "Campinas", destino_c)
print(f"  (Comparacao) Campinas -> Brasilia: {custo_c2} km | {' -> '.join(caminho_c2)}")

Cenario 2 — Logistica de Entregas
  Origem:  Sao Paulo
  Destino: Brasilia

  Rota com menor distancia:
  Sao Paulo -> Ribeirao Preto -> Uberlandia -> Goiania -> Brasilia
  Distancia total: 915 km

  (Comparacao) Campinas -> Brasilia: 825 km | Campinas -> Ribeirao Preto -> Uberlandia -> Goiania -> Brasilia


### Cenário 3 — Grade Curricular com Pré-requisitos

Em um curso de Engenharia de Software, cada disciplina exige a conclusão de matérias anteriores.
O grafo modela as **dependências entre disciplinas**, onde o peso representa a **carga horária** da disciplina predecessora que precisa ser concluída.

Objetivo: encontrar o caminho de menor carga acumulada de pré-requisitos entre **Algoritmos** e **TCC**.

In [5]:
# Grafo de dependências da grade curricular
# Pesos = carga horária da disciplina de origem necessária para liberar a próxima
grafo_grade = {
    "Algoritmos":              {"Estrutura de Dados": 60, "Logica": 40},
    "Logica":                  {"Matematica Discreta": 40, "Estrutura de Dados": 40},
    "Matematica Discreta":     {"Grafos": 60, "Probabilidade": 60},
    "Estrutura de Dados":      {"Grafos": 60, "Banco de Dados": 60, "POO": 60},
    "POO":                     {"Engenharia de Software": 60, "Padroes de Projeto": 60},
    "Padroes de Projeto":      {"Arquitetura de Software": 60},
    "Grafos":                  {"Inteligencia Artificial": 60, "Algoritmos Avancados": 80},
    "Probabilidade":           {"Inteligencia Artificial": 60},
    "Banco de Dados":          {"Engenharia de Software": 60},
    "Engenharia de Software":  {"TCC": 80},
    "Arquitetura de Software": {"TCC": 80},
    "Inteligencia Artificial": {"TCC": 80},
    "Algoritmos Avancados":    {"TCC": 80},
    "TCC":                     {},
}

origem_g = "Algoritmos"
destino_g = "TCC"
custo_g, caminho_g = dijkstra(grafo_grade, origem_g, destino_g)

print("=" * 60)
print("Cenario 3 — Grade Curricular")
print("=" * 60)
print(f"  Origem:  {origem_g}")
print(f"  Destino: {destino_g}")
print()
print("  Caminho com menor carga acumulada de pre-requisitos:")
print("  " + " -> ".join(caminho_g))
print(f"  Carga acumulada: {custo_g} horas")

Cenario 3 — Grade Curricular
  Origem:  Algoritmos
  Destino: TCC

  Caminho com menor carga acumulada de pre-requisitos:
  Algoritmos -> Estrutura de Dados -> Banco de Dados -> Engenharia de Software -> TCC
  Carga acumulada: 260 horas


### Conclusão — Quando usar Grafo Direcionado + Dijkstra?

| Requisito | Descrição |
|-----------|-----------|
| **Relação de precedência** | Há etapas ou lugares que só são alcançados via outros |
| **Custos heterogêneos** | Cada transição tem um custo diferente (tempo, distância, esforço) |
| **Pesos ≥ 0** | Dijkstra exige pesos não-negativos; para negativos use Bellman-Ford |
| **Solução ótima** | É necessário garantir o menor custo total, não apenas uma solução viável |

Os três cenários acima e o CRM compartilham exatamente essa estrutura:
um conjunto de **estados** (nós) com **transições custosas** (arestas), onde queremos minimizar o custo total do percurso.